In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
from itertools import combinations 

base_dir    = '/Users/lilymarino/Documents/GitHub/t-c_continuum'
df          = pd.read_csv(os.path.join(base_dir, 'data', 'dataset-ariWave1_data-allPhysio_detail-standardizedChangeScores.csv'))
ref_means   = pd.read_csv(os.path.join(base_dir, 'ref_data', 'dataset-arb_data-allPhysio_detail-2clusterStandardizedChangeScores.csv'))

     subject  trial  SCR_delta  SCL_delta  RR_delta  IBI_delta  RSA_delta  \
0    sub-003      1  -1.108376  -0.680116  0.400899  -0.413194   0.313878   
1    sub-003      2  -0.072717  -0.725578 -0.555781   1.064173  -0.029487   
2    sub-003      3   0.617723  -0.752049  1.221752   0.166406  -0.345952   
3    sub-003      4   0.272503  -0.658250  0.032589   2.033700   0.449909   
4    sub-006      1   0.893899   0.766347  0.799265  -2.586264  -4.264711   
..       ...    ...        ...        ...       ...        ...        ...   
179  sub-067      4   1.377207   0.680878  1.221955  -0.121121   0.098942   
180  sub-068      1   0.272503   0.414029 -0.235272  -1.695569   1.313430   
181  sub-068      2  -0.417936   0.032347  0.449646  -1.141431   0.421705   
182  sub-068      3  -0.763156   0.056894  0.116335  -0.560364   1.832569   
183  sub-068      4  -0.763156  -0.112534 -0.245371  -1.249960   0.728927   

     MAP_delta  LVET_delta  CO_delta  PEP_delta  TPR_delta  
0     0.472260

In [ ]:
# all available features:
        # SCR       # PEP
        # SCL       # LVET
        # IBI       # CO
        # RR        # MAP
        # RSA       # TPR

### NEED TO DECIDE ON THIS - EXCLUDING RR AND LVET FOR NOW (NOT AS RELEVANT?) ###
all_relevant_metrics = ['SCR_delta', 'SCL_delta', 'IBI_delta', 'RSA_delta', 
                         'MAP_delta', 'CO_delta', 'PEP_delta', 'TPR_delta']

# calculate number of combinations
# Generate all combinations
all_metric_combos = []
for n in range(6, 8):  # 6 and 7 variables
    combos = list(combinations(all_relevant_metrics, n))
    all_metric_combos.extend(combos)


print(f"\nTotal combinations: {len(all_metric_combos)}")

# loop through each subset of features
for combo_idx, metrics in enumerate(all_metric_combos):
    
    # name each combination (may want to edit this)
    subset_name = f"combo_{combo_idx + 1}_n{len(metrics)}"
    metrics = list(metrics)  # Convert tuple to list
    print(f"Processing: {subset_name}")
    results_subset = []
    
    # calculate euclidean distances per subject per trial
    for index, row in df.iterrows():
        subject = row['subject']
        trial   = row['trial']
        
        delta_values    = row[metrics].values
        trial_means     = ref_means[ref_means['trial'] == trial]
        feature_names   = [f.replace('_delta', '') for f in metrics]
        trial_means_filtered = trial_means[trial_means['metric'].isin(feature_names)]
        
        challenge_means = trial_means_filtered['challenge_cluster_means'].values
        threat_means = trial_means_filtered['threat_cluster_means'].values
        
        # calculate euclidean distance between subject per trial and reference per trial mean change scores
        challenge_dist  = euclidean(delta_values, challenge_means)
        threat_dist     = euclidean(delta_values, threat_means)
        
        results_subset.append({
            'subject': subject,
            'trial': trial,
            'challenge_dist': challenge_dist,
            'threat_dist': threat_dist
        })
    
    results_subset = pd.DataFrame(results_subset)
    
    # calculate similarity scores (1 / distance scores)
    results_subset['challenge_sim'] = 1 / results_subset['challenge_dist']
    results_subset['threat_sim'] = 1 / results_subset['threat_dist']
    results_subset['challenge_sim_norm'] = results_subset['challenge_sim'] / (results_subset['challenge_sim'] + results_subset['threat_sim'])
    results_subset['threat_sim_norm'] = results_subset['threat_sim'] / (results_subset['challenge_sim'] + results_subset['threat_sim'])
    
    # plot data
    subjects = results_subset['subject'].unique()
    n_subjects = len(subjects)
    n_cols = int(np.ceil(np.sqrt(n_subjects)))
    n_rows = int(np.ceil(n_subjects / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize = (n_cols * 3, n_rows * 2.5))
    axes = axes.flatten()
    fig.suptitle(f'Cluster Similarity: {subset_name}', fontsize=16, y=0.995)
    
    for idx, subject in enumerate(subjects):
        
        sub_data = results_subset[results_subset['subject'] == subject]
        
        axes[idx].plot(sub_data['trial'], sub_data['challenge_sim_norm'], marker='o', color='#482173', linewidth=1.5)
        axes[idx].plot(sub_data['trial'], sub_data['threat_sim_norm'], marker='s', color='#29af7f', linewidth=1.5)
        axes[idx].set_title(subject, fontsize=9)
        axes[idx].set_xticks([1, 2, 3, 4])
        axes[idx].set_ylim([0, 1])
        axes[idx].grid(True, alpha=0.3)
        
        if idx >= len(subjects) - n_cols:
            axes[idx].set_xlabel('Trial', fontsize=8)

        if idx % n_cols == 0:
            axes[idx].set_ylabel('Cluster Similarity', fontsize=8)
    
    for idx in range(n_subjects, len(axes)):
        axes[idx].axis('off')
    
    fig.legend(['Challenge', 'Threat'], loc = 'upper right', 
               bbox_to_anchor=(0.98, 0.98), fontsize=10)
    
    plt.tight_layout()
    output_path = os.path.join(base_dir, 'plots', f'similarity_{subset_name.replace(" ", "_")}.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
print("Complete")


Total combinations: 36
Processing: combo_1_n6
Saved: /Users/lilymarino/Documents/GitHub/t-c_continuum/plots/similarity_combo_1_n6.png
Processing: combo_2_n6
Saved: /Users/lilymarino/Documents/GitHub/t-c_continuum/plots/similarity_combo_2_n6.png
Processing: combo_3_n6
Saved: /Users/lilymarino/Documents/GitHub/t-c_continuum/plots/similarity_combo_3_n6.png
Processing: combo_4_n6
Saved: /Users/lilymarino/Documents/GitHub/t-c_continuum/plots/similarity_combo_4_n6.png
Processing: combo_5_n6
Saved: /Users/lilymarino/Documents/GitHub/t-c_continuum/plots/similarity_combo_5_n6.png
Processing: combo_6_n6
Saved: /Users/lilymarino/Documents/GitHub/t-c_continuum/plots/similarity_combo_6_n6.png
Processing: combo_7_n6
Saved: /Users/lilymarino/Documents/GitHub/t-c_continuum/plots/similarity_combo_7_n6.png
Processing: combo_8_n6
Saved: /Users/lilymarino/Documents/GitHub/t-c_continuum/plots/similarity_combo_8_n6.png
Processing: combo_9_n6
Saved: /Users/lilymarino/Documents/GitHub/t-c_continuum/plots/sim